# Harness marginal predictions with 95% confidence intervals

This notebook operates on the fitted binomial mixed-effects logistic model:

`Defense ~ Harness * Model + AttackCategory + (1 | Task)`

Using：

`emmeans(fit, ~ harness, type='response', weights='equal')`

we report the marginal prediction, standard error, and 95% confidence interval for each harness, computed as an equally weighted average across all models and attack categories.

In [1]:
import os
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)

## 1. Locate Converged GLMM and py310 R Environment

In [2]:
def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'web-agent/harness_security').exists():
            return candidate
    raise FileNotFoundError('Cannot locate repository root')

REPO_ROOT = find_repo_root()
MODEL_PATH = (
    REPO_ROOT / 'web-agent/harness_security/'
    'analysis_task_only_output/main_model_glmer.rds'
)
OUTPUT_DIR = (
    REPO_ROOT / 'web-agent/harness_security/'
    'analysis_emmeans_confidence_output'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RSCRIPT = Path(os.getenv(
    'PY310_RSCRIPT', '/opt/homebrew/anaconda3/envs/py310/bin/Rscript'))

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found: {MODEL_PATH}. Run analysis_task_only.ipynb first.')
if not RSCRIPT.exists():
    raise FileNotFoundError(f'Rscript not found: {RSCRIPT}')

print('Model:', MODEL_PATH)
print('Output:', OUTPUT_DIR)
print('Rscript:', RSCRIPT)

Model: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_task_only_output/main_model_glmer.rds
Output: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_emmeans_confidence_output
Rscript: /opt/homebrew/anaconda3/envs/py310/bin/Rscript


## 2. Compute Harness Marginal Predictions Using emmeans

95% CIs are asymptotic Wald intervals derived from the covariance matrix of the model’s fixed effects, and are back-transformed from the logit scale to the probability scale.

In [3]:
R_CODE = r'''
args <- commandArgs(trailingOnly=TRUE)
model_path <- args[[1]]
output_dir <- args[[2]]

suppressPackageStartupMessages({
  library(lme4)
  library(emmeans)
  library(car)
})

fit <- readRDS(model_path)
expected_formula <- 'defense ~ harness * backend_model + attack_category + (1 | task_id)'
actual_formula <- paste(deparse(formula(fit)), collapse=' ')
if (actual_formula != expected_formula) {
  stop(paste('Unexpected model formula:', actual_formula))
}
messages <- fit@optinfo$conv$lme4$messages
if (!is.null(messages)) stop(paste(messages, collapse='; '))
if (isSingular(fit, tol=1e-4)) stop('Model is singular')

task_variance <- as.data.frame(VarCorr(fit))$vcov[1]
model_info <- data.frame(
  formula=actual_formula,
  n=nobs(fit),
  converged=TRUE,
  singular=FALSE,
  task_variance=task_variance,
  task_sd=sqrt(task_variance),
  task_latent_icc=task_variance/(task_variance + pi^2/3)
)
write.csv(model_info, file.path(output_dir, 'model_info.csv'), row.names=FALSE)

emm <- emmeans(
  fit, ~ harness, type='response', weights='equal', level=0.95
)
emm_df <- as.data.frame(summary(emm, infer=c(TRUE, FALSE), level=0.95))
names(emm_df)[names(emm_df) == 'prob'] <- 'marginal_probability'
names(emm_df)[names(emm_df) == 'asymp.LCL'] <- 'ci_low_95'
names(emm_df)[names(emm_df) == 'asymp.UCL'] <- 'ci_high_95'
write.csv(emm_df, file.path(output_dir, 'harness_marginal_predictions_95ci.csv'), row.names=FALSE)

omnibus <- as.data.frame(Anova(fit, type=3, test.statistic='Chisq'))
omnibus$source <- rownames(omnibus)
rownames(omnibus) <- NULL
omnibus <- omnibus[,c('source', setdiff(names(omnibus), 'source'))]
write.csv(omnibus, file.path(output_dir, 'omnibus_type3.csv'), row.names=FALSE)

cat('Formula:', actual_formula, '\n')
cat('N:', nobs(fit), '\n')
print(emm_df)
'''

run = subprocess.run(
    [str(RSCRIPT), '-e', R_CODE, str(MODEL_PATH), str(OUTPUT_DIR)],
    text=True, capture_output=True,
)
print(run.stdout)
if run.stderr:
    print('R messages/warnings:\n', run.stderr)
if run.returncode != 0:
    raise RuntimeError(f'R analysis failed with exit code {run.returncode}')

Formula: defense ~ harness * backend_model + attack_category + (1 | task_id) 
N: 15783 
 harness        marginal_probability         SE  df ci_low_95 ci_high_95
 seeact                    0.7386953 0.03144488 Inf 0.6725868  0.7955140
 browser_text              0.7742767 0.02722702 Inf 0.7165242  0.8231671
 browser_vision            0.7458366 0.02914073 Inf 0.6846521  0.7986414
 agent_e                   0.8969726 0.01526558 Inf 0.8629801  0.9232817
 skyvern_1                 0.8275013 0.02270736 Inf 0.7783769  0.8675893
 skyvern_2                 0.6206268 0.03590388 Inf 0.5481832  0.6880645

Results are averaged over the levels of: backend_model, attack_category 
Confidence level used: 0.95 
Intervals are back-transformed from the logit scale 

R messages/warnings:
 NOTE: Results may be misleading due to involvement in interactions



## 3. Confidence Interval Table

In [4]:
model_info = pd.read_csv(OUTPUT_DIR / 'model_info.csv')
marginal = pd.read_csv(OUTPUT_DIR / 'harness_marginal_predictions_95ci.csv')
omnibus = pd.read_csv(OUTPUT_DIR / 'omnibus_type3.csv')

display(model_info)
display(marginal)
display(omnibus)

assert bool(model_info.loc[0, 'converged'])
assert not bool(model_info.loc[0, 'singular'])
assert len(marginal) == 6
assert marginal['marginal_probability'].between(0, 1).all()
assert marginal['ci_low_95'].between(0, 1).all()
assert marginal['ci_high_95'].between(0, 1).all()

,formula,n,converged,singular,task_variance,task_sd,task_latent_icc
0,defense ~ harness * backend_model + attack_cat...,15783,True,False,9.94603,3.153733,0.751444


,harness,marginal_probability,SE,df,ci_low_95,ci_high_95
0,seeact,0.738695,0.031445,inf,0.672587,0.795514
1,browser_text,0.774277,0.027227,inf,0.716524,0.823167
2,browser_vision,0.745837,0.029141,inf,0.684652,0.798641
3,agent_e,0.896973,0.015266,inf,0.862980,0.923282
4,skyvern_1,0.827501,0.022707,inf,0.778377,0.867589
5,skyvern_2,0.620627,0.035904,inf,0.548183,0.688065


,source,Chisq,Df,Pr(>Chisq)
0,(Intercept),75.212291,1,4.227273e-18
1,harness,331.173121,5,1.974696e-69
2,backend_model,437.632057,6,2.251666e-91
3,attack_category,4.812647,2,9.014609e-02
4,harness:backend_model,299.813178,30,2.882264e-46


## 4. Auto-generate Markdown Confidence Report

In [5]:
def p_text(value):
    if value < 0.001:
        return f'{value:.2e}'
    return f'{value:.3f}'

lines = [
    '# Mixed-effects logistic confidence report',
    '',
    '## Model',
    '',
    '`Defense ~ Harness * Model + AttackCategory + (1 | Task)`',
    '',
    f"- Observations: {int(model_info.loc[0, 'n']):,}",
    f"- Task random-intercept SD: {model_info.loc[0, 'task_sd']:.3f}",
    f"- Task latent-scale ICC: {model_info.loc[0, 'task_latent_icc']:.3f}",
    '- Estimation: binomial-logit GLMM, Laplace approximation.',
    '- Intervals: asymptotic 95% Wald confidence intervals, back-transformed to probability.',
    '- Marginalization: equal weights over backend model and attack category.',
    '',
    '## Harness marginal predictions',
    '',
    '| Harness | Marginal prediction | 95% CI |',
    '|---|---:|---:|',
]
for row in marginal.itertuples(index=False):
    lines.append(
        f'| {row.harness} | {100*row.marginal_probability:.2f}% | '
        f'[{100*row.ci_low_95:.2f}%, {100*row.ci_high_95:.2f}%] |'
    )

lines.extend([
    '',
    '## Omnibus fixed-effect tests',
    '',
    '| Source | Wald chi-square | df | p-value |',
    '|---|---:|---:|---:|',
])
for _, row in omnibus.iterrows():
    lines.append(
        f"| {row['source']} | {row['Chisq']:.2f} | "
        f"{int(row['Df'])} | {p_text(row['Pr(>Chisq)'])} |"
    )

lines.extend([
    '',
    '## Interpretation note',
    '',
    'Harness predictions are averaged equally over all backend models and attack categories. '
    'Because the Harness×Model interaction is present, these overall marginal predictions '
    'summarize average harness behavior and do not imply that the same harness ordering holds '
    'for every backend model.',
])

REPORT_PATH = OUTPUT_DIR / 'confidence_report.md'
REPORT_PATH.write_text('\n'.join(lines), encoding='utf-8')
display(Markdown(REPORT_PATH.read_text(encoding='utf-8')))
print('Saved report:', REPORT_PATH)

# Mixed-effects logistic confidence report

## Model

`Defense ~ Harness * Model + AttackCategory + (1 | Task)`

- Observations: 15,783
- Task random-intercept SD: 3.154
- Task latent-scale ICC: 0.751
- Estimation: binomial-logit GLMM, Laplace approximation.
- Intervals: asymptotic 95% Wald confidence intervals, back-transformed to probability.
- Marginalization: equal weights over backend model and attack category.

## Harness marginal predictions

| Harness | Marginal prediction | 95% CI |
|---|---:|---:|
| seeact | 73.87% | [67.26%, 79.55%] |
| browser_text | 77.43% | [71.65%, 82.32%] |
| browser_vision | 74.58% | [68.47%, 79.86%] |
| agent_e | 89.70% | [86.30%, 92.33%] |
| skyvern_1 | 82.75% | [77.84%, 86.76%] |
| skyvern_2 | 62.06% | [54.82%, 68.81%] |

## Omnibus fixed-effect tests

| Source | Wald chi-square | df | p-value |
|---|---:|---:|---:|
| (Intercept) | 75.21 | 1 | 4.23e-18 |
| harness | 331.17 | 5 | 1.97e-69 |
| backend_model | 437.63 | 6 | 2.25e-91 |
| attack_category | 4.81 | 2 | 0.090 |
| harness:backend_model | 299.81 | 30 | 2.88e-46 |

## Interpretation note

Harness predictions are averaged equally over all backend models and attack categories. Because the Harness×Model interaction is present, these overall marginal predictions summarize average harness behavior and do not imply that the same harness ordering holds for every backend model.

Saved report: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_emmeans_confidence_output/confidence_report.md


## 5. Outputs

- `confidence_report.md`
- `harness_marginal_predictions_95ci.csv`
- `omnibus_type3.csv`
- `model_info.csv`